<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">

# Accelerating End-to-End Data Science Workflows # 

## 05 - Grouping ##

**Table of Contents**
<br>
This notebook discusses and demonstrates how grouping in used in data science. This notebook covers the below sections: 
1. [Grouping](#Grouping)
    * [Split, Apply, and Combine](#Split,-Apply,-and-Combine)
    * [Exercise #1 - Average Age Per County](#Exercise-#1---Average-Age-Per-County)
2. [Binning](#Binning)
    * [Exercise #2 - Using the Profiler](#Exercise-#2---Using-the-Profiler)
3. [Advanced Groupby Operations](#Advanced-Groupby-Operations)
    * [`.apply()`](#.apply())
    * [`.transform()`](#.transform())
4. [Pivot Table](#Pivot-Table)

## Grouping ##
In data science, we often would like to split data into groups and perform further analysis on them such as: 
* Aggregate based on the grouping
* Compare metrics across different groups
* Understand patterns in data across different groups
* Remove duplicates or fill missing values based on group-level information
* Create new features based on group-level statistics
* Integrate with visualization

Below we load in our dataset. 

In [1]:
# DO NOT CHANGE THIS CELL
# %load_ext cudf.pandas
import pandas as pd
import time

In [ ]:
# DO NOT CHANGE THIS CELL
dtype_dict={
    'age': 'int8', 
    'sex': 'category', 
    'county': 'category', 
    'lat': 'float32', 
    'long': 'float32', 
    'name': 'category'
}
        
df=pd.read_csv('./data/uk_pop.csv', dtype=dtype_dict)
df.head()

In [6]:
import polars as pl

df = pl.DataFrame({
    "city": ["London", "London", "Bristol", "Bristol", "Bristol", "Manchester", "Manchester"],
    "temperature": [15, 17, 14, 13, 16, 18, 19],
    "humidity": [55, 60, 65, 63, 64, 50, 48],
    "rainy_days": [12, 9, 15, 18, 17, 10, 8]
})
df

city,temperature,humidity,rainy_days
str,i64,i64,i64
"""London""",15,55,12
"""London""",17,60,9
"""Bristol""",14,65,15
"""Bristol""",13,63,18
"""Bristol""",16,64,17
"""Manchester""",18,50,10
"""Manchester""",19,48,8


In [8]:
df.group_by("city").agg([
    pl.col("temperature").mean().alias("avg_temp"),
    pl.col("temperature").max().alias("max_temp"),
    pl.col("temperature").min().alias("min_temp"),
    pl.len().alias("record_count")
])



city,avg_temp,max_temp,min_temp,record_count
str,f64,i64,i64,u32
"""Manchester""",18.5,19,18,2
"""London""",16.0,17,15,2
"""Bristol""",14.333333,16,13,3


In [9]:
df.group_by("city").agg(pl.len().alias("size"))


city,size
str,u32
"""Manchester""",2
"""Bristol""",3
"""London""",2


In [10]:
df.select(pl.cov("temperature", "humidity"))

temperature
f64
-11.666667


In [11]:
%%time
df = df.with_columns(pl.Series("growth_rate", [1.02, 1.01, 0.99, 1.03, 1.04, 1.00, 1.05]))

df = df.with_columns([
    pl.col("growth_rate").cum_prod().alias("total_growth"),
    pl.col("rainy_days").cum_sum().alias("rain_total")
])

df


CPU times: total: 0 ns
Wall time: 0 ns


city,temperature,humidity,rainy_days,growth_rate,total_growth,rain_total
str,i64,i64,i64,f64,f64,i64
"""London""",15,55,12,1.02,1.02,12
"""London""",17,60,9,1.01,1.0302,21
"""Bristol""",14,65,15,0.99,1.019898,36
"""Bristol""",13,63,18,1.03,1.050495,54
"""Bristol""",16,64,17,1.04,1.092515,71
"""Manchester""",18,50,10,1.0,1.092515,81
"""Manchester""",19,48,8,1.05,1.14714,89


In [15]:
import polars as pl

df = pl.DataFrame({
    "region": ["North", "North", "South", "South", "East", "East"],
    "sales": [1000, 1500, 2000, 1800, 1200, 1300],
    "units": [10, 15, 20, 18, 12, 13]
})

summary = df.group_by("region").agg([
    pl.col("sales").sum().alias("total_sales"),
    pl.col("sales").mean().alias("avg_sales"),
    pl.col("units").sum().alias("total_units"),
    pl.col("units").mean().alias("avg_units")
])

summary


region,total_sales,avg_sales,total_units,avg_units
str,i64,f64,i64,f64
"""East""",2500,1250.0,25,12.5
"""South""",3800,1900.0,38,19.0
"""North""",2500,1250.0,25,12.5


In [20]:
df = pl.DataFrame({
    "region": ["North", "North", "South", "South", "East", "East"],
    "sales": [1000, 1500, 2000, 1800, 1200, 1300],
    "units": [10, 15, 20, 18, 12, 13]
})

result = (
    df.group_by("region")
      .agg([
          pl.col("sales").mean().alias("avg_sales"),
          pl.when(pl.col("sales") > 1300)
            .then(pl.col("sales"))
            .otherwise(None)
            .mean()
            .alias("high_value_avg")
      ])
)
print(result)


shape: (3, 3)
┌────────┬───────────┬────────────────┐
│ region ┆ avg_sales ┆ high_value_avg │
│ ---    ┆ ---       ┆ ---            │
│ str    ┆ f64       ┆ f64            │
╞════════╪═══════════╪════════════════╡
│ East   ┆ 1250.0    ┆ null           │
│ North  ┆ 1250.0    ┆ 1500.0         │
│ South  ┆ 1900.0    ┆ 1900.0         │
└────────┴───────────┴────────────────┘


In [23]:
df.with_columns([
    pl.col('units').cast(pl.Int16),
    pl.col('sales').cast(pl.Int32),
])


region,sales,units
str,i32,i16
"""North""",1000,10
"""North""",1500,15
"""South""",2000,20
"""South""",1800,18
"""East""",1200,12
"""East""",1300,13


In [25]:
df = df.with_columns([
    pl.col(col_name).cast(dtype) for col_name, dtype in {
        "units": pl.UInt8,
        "sales": pl.UInt16
    }.items()
])
df

region,sales,units
str,u16,u8
"""North""",1000,10
"""North""",1500,15
"""South""",2000,20
"""South""",1800,18
"""East""",1200,12
"""East""",1300,13


In [21]:
df.head(5)

region,sales,units
str,i64,i64
"""North""",1000,10
"""North""",1500,15
"""South""",2000,20
"""South""",1800,18
"""East""",1200,12


In [2]:
import pandas as pd
df = pd.DataFrame({
    "city": ["London", "London", "Bristol", "Bristol", "Bristol", "Manchester", "Manchester"],
    "temperature": [15, 17, 14, 13, 16, 18, 19],
    "humidity": [55, 60, 65, 63, 64, 50, 48],
    "rainy_days": [12, 9, 15, 18, 17, 10, 8]
})

In [5]:
%%time
df['growth_rate'] = [1.02, 1.01, 0.99, 1.03, 1.04, 1.00, 1.05]
df['total_growth'] = df['growth_rate'].cumprod()
df[['growth_rate', 'total_growth']]
df['rain_total'] = df['rainy_days'].cumsum()
df[['rainy_days', 'rain_total']]
df


CPU times: total: 0 ns
Wall time: 1.75 ms


,city,temperature,humidity,rainy_days,growth_rate,total_growth,rain_total
0,London,15,55,12,1.02,1.020000,12
1,London,17,60,9,1.01,1.030200,21
2,Bristol,14,65,15,0.99,1.019898,36
3,Bristol,13,63,18,1.03,1.050495,54
4,Bristol,16,64,17,1.04,1.092515,71
5,Manchester,18,50,10,1.00,1.092515,81
6,Manchester,19,48,8,1.05,1.147140,89


In [12]:
import polars as pl
import numpy as np

INT_TYPES = [
    (pl.Int8, np.iinfo(np.int8).min, np.iinfo(np.int8).max),
    (pl.Int16, np.iinfo(np.int16).min, np.iinfo(np.int16).max),
    (pl.Int32, np.iinfo(np.int32).min, np.iinfo(np.int32).max),
    (pl.Int64, np.iinfo(np.int64).min, np.iinfo(np.int64).max),
]

FLOAT_TYPES = [
    (pl.Float32, np.finfo(np.float32).min, np.finfo(np.float32).max),
    (pl.Float64, np.finfo(np.float64).min, np.finfo(np.float64).max)
]


In [13]:
def optimize_dtypes(df: pl.DataFrame) -> pl.DataFrame:
    """
    Automatically downcast numeric columns to the smallest safe data type.
    """
    new_columns = []
    for col_name, dtype in zip(df.columns, df.dtypes):
        series = df[col_name]
        
        # Integer columns
        if dtype in [pl.Int64, pl.Int32, pl.Int16, pl.Int8]:
            col_min, col_max = series.min(), series.max()
            for t, t_min, t_max in INT_TYPES:
                if t_min <= col_min and col_max <= t_max:
                    new_columns.append(series.cast(t).alias(col_name))
                    break
        # Float columns
        elif dtype in [pl.Float64, pl.Float32]:
            col_min, col_max = series.min(), series.max()
            for t, t_min, t_max in FLOAT_TYPES:
                if t_min <= col_min and col_max <= t_max:
                    new_columns.append(series.cast(t).alias(col_name))
                    break
        else:
            # Non-numeric columns remain unchanged
            new_columns.append(series)
            
    return pl.DataFrame(new_columns)


In [15]:
%%time
#Eager method
df = pl.DataFrame({
    "big_int": [100, 200, 300],
    "medium_int": [2_000_000, 3_000_000, 1_500_000],
    "big_float": [1.5e10, 2.5e10, 3.0e10]
})

df_optimized = optimize_dtypes(df)
print(df_optimized.dtypes)


[Int16, Int32, Float32]
CPU times: total: 0 ns
Wall time: 0 ns


## Split, Apply, and Combine ##
We use the `.groupby()` method to to group large amounts of data and compute operations on these groups. A groupby operation involves some combination of splitting the object, applying a function, and combining the results. cuDF implements record grouping in a manner comparable to Pandas, but with some notable differences. 

<p><img src='images/groupby.png' width=720></p>

cuDF supports a number of common `DataFrameGroupBy` computations and descriptive statistics, such as `.size()`, `.mean()`, `.count()`, `.cov()`, `.cumprod()`, `.cumsum()`, `.max()`, `.min()`, `.nunique()`. 

**Note**: More information about how `.groupby()` behaves for pandas and how it differs from cuDF can be found in the links below: 
* [pandas](https://pandas.pydata.org/docs/user_guide/groupby.html)
* [cuDF](https://docs.rapids.ai/api/cudf/stable/user_guide/groupby/)

Below we find the number of people in each county. 

In [ ]:
# DO NOT CHANGE THIS CELL
df.groupby('county').size()

**Note**: The results is unsorted. We can sort the output using the `.sort_index()` or `.sort_values()` method. 

Below we count the number of people with the most and least popular names. 

In [ ]:
# DO NOT CHANGE THIS CELL
df.groupby('name').size().sort_values()

Below we find the approximate centers of each county using `.groupby().mean()`. When performing groupby operations, we should **only** include columns that are being used. 

In [ ]:
%%cudf.pandas.line_profile
# DO NOT CHANGE THIS CELL

county_center_df=df[['county', 'lat', 'long']].groupby('county')[['lat', 'long']].mean()
display(county_center_df)

In [ ]:
# DO NOT CHANGE THIS CELL
county_center_df.columns=['lat_county_center', 'long_county_center']
county_center_df.to_csv('county_centroid.csv')

### Exercise #1 - Average Age Per County ###
We would like to find the average age for each county. We will need to use both `.groupby()` and `.sort_values()`. Using the `.mean()` method on the data grouped by `county`, identify the 5 counties with the highest average age. 

**Instructions**: <br>
* Modify the `<FIXME>` only and execute the below cell find the average age for each county. 

In [ ]:
df[['county', 'age']].groupby(<<<<FIXME>>>>)['age']\
                     .<<<<FIXME>>>>()\
                     .sort_values(ascending=False)\
                     .head()

In [2]:
a = 1 + 2 + \
    3 + 4
a

10

Click ... for solution. 

## Binning ##
When grouping continuous numerical data, it is sometimes helpful to bin numbers into discrete intervals or buckets. There are primarily two ways of binning: 
* Equal-width binning: divide the range into equal-sized intervals
* Custom binning: define custom bins based on domain knowledge or specific criteria

The `.cut()` function can be used to bin values into discrete intervals

In [ ]:
%%cudf.pandas.line_profile
# DO NOT CHANGE THIS CELL

bins=[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
df['age_bucket']=pd.cut(df['age'].values, bins=bins, right=True, include_lowest=True, labels=False)
display(df.groupby('age_bucket').size())

### Exercise #2 - Using the Profiler ###
cuDF pandas will attempt to use the GPU whenever possible and fall back to CPU for certain operations. Running the code with the `cudf.pandas.line_profile` magic command generates a report showing which operations used the GPU and which used the CPU. 

**Instructions**: <br>
* Notice that the below cell is a very similar operation as before, except that it uses the `range()` function for the `bins` parameter. As it stands, this is not supported in cuDF. 
* Execute the cell below to run the binning operation on the CPU.
* Compare the time it takes to run the similar operation above. 

In [ ]:
%%cudf.pandas.line_profile
# DO NOT CHANGE THIS CELL

df['age_bucket']=pd.cut(df['age'].values, bins=range(0, 100, 10), right=True, include_lowest=True, labels=False)
display(df.groupby('age_bucket').size())

**Note**: The profiler can help us identify parts of our code that could be rewritten to be more GPU-friendly. 

## Advanced Groupby Operations ##
We can also use function application helpers on `DataFrameGroupBy` instances: 
* `DataFrameGroupby.aggregate()` / `Groupby.agg()`(alias): used when we have specific computation for different columns or more than one computation on the same column
* `DataFrameGroupby.apply()`: used when we want to perform a specific user-defined function to each group
* `DataFrameGroupby.transform()`: used when the resulting values should be broadcast across the whole group and return a same-indexed dataframe

### `.apply()` ###
The `.apply()` method will **sequentially** apply the function group-wise and concatenate the results together. We can pass a callable function to be performed on the entire DataFrame for each group. 

Below we calculate the distance of each person from their respective county center. 

In [ ]:
import numpy as np

def distance_from_center(df_group):
    lat_center = df_group["lat"].mean()
    long_center = df_group["long"].mean()
    df_group = df_group.with_columns([
        ((pl.col("lat") - lat_center)**2 + (pl.col("long") - long_center)**2).sqrt().alias("distance")
    ])
    return df_group

# Example DataFrame
df2 = pl.DataFrame({
    "county": ["A", "A", "B", "B"],
    "lat": [51.5, 51.6, 52.5, 52.6],
    "long": [-0.1, -0.2, -1.0, -1.1]
})

# Group and apply custom function
df_result = df2.group_by("county", maintain_order=True).map_groups(distance_from_center)
print(df_result)


shape: (4, 4)
┌────────┬──────┬──────┬──────────┐
│ county ┆ lat  ┆ long ┆ distance │
│ ---    ┆ ---  ┆ ---  ┆ ---      │
│ str    ┆ f64  ┆ f64  ┆ f64      │
╞════════╪══════╪══════╪══════════╡
│ A      ┆ 51.5 ┆ -0.1 ┆ 0.070711 │
│ A      ┆ 51.6 ┆ -0.2 ┆ 0.070711 │
│ B      ┆ 52.5 ┆ -1.0 ┆ 0.070711 │
│ B      ┆ 52.6 ┆ -1.1 ┆ 0.070711 │
└────────┴──────┴──────┴──────────┘


In [ ]:
# DO NOT CHANGE THIS CELL

# define distance function
def distance(lat_1, long_1, lat_2, long_2): 
    return ((lat_2-lat_1)**2+(long_2-long_1)**2)**0.5

In [ ]:
%%cudf.pandas.line_profile
# DO NOT CHANGE THIS CELL

distance_df=df.groupby('county')[['lat', 'long']].apply(lambda x: distance(x['lat'], x['long'], x['lat'].mean(), x['long'].mean()))
df['R_1']=distance_df.reset_index(level=0, drop=True)

We can also define the function in-line. 

In [ ]:
%%cudf.pandas.line_profile
# DO NOT CHANGE THIS CELL

df['R_2']=df.groupby('county')[['lat', 'long']].apply(lambda x: ((x['lat'].mean()-x['lat'])**2+(x['long'].mean()-x['long'])**2)**0.5).reset_index(level=0, drop=True)

**Note**: This is quite slow due to the iterative nature of the `.apply()` method. 

### `.transform()` ###
The `.transform()` method aggregates each group, and broadcasts the result to the group size, resuliting in a DataFrame that is the same size and index as the input DataFrame. Underneath the hood, the `.transform()` method passes each column individually as a Series to the function. 

Below we group the DataFrame by `county` and transform the columns `lat` and `long` using `mean`. We will subtract the transformed mean from the original columns, then apply the distance formula to calculate the resulting distance.  By keeping the DataFrame the same shape, we can perform cuDF operations quickly, resulting in performance gain. 

In [21]:
# polars have over() where pandas have transform()

df2 = df2.with_columns([
    pl.col("lat").mean().over("county").alias("lat_center"),
    pl.col("long").mean().over("county").alias("long_center")
])

# Compute distance from group center
df2 = df2.with_columns([
    ((pl.col("lat") - pl.col("lat_center"))**2 + (pl.col("long") - pl.col("long_center"))**2).sqrt().alias("distance")
])

df2


county,lat,long,lat_center,long_center,distance
str,f64,f64,f64,f64,f64
"""A""",51.5,-0.1,51.55,-0.15,0.070711
"""A""",51.6,-0.2,51.55,-0.15,0.070711
"""B""",52.5,-1.0,52.55,-1.05,0.070711
"""B""",52.6,-1.1,52.55,-1.05,0.070711


In [ ]:
# DO NOT CHANGE THIS CELL
# make data types more precise
df[['lat', 'long']]=df[['lat', 'long']].astype('float64')

In [ ]:
%%cudf.pandas.line_profile
# DO NOT CHANGE THIS CELL

c=['lat', 'long']
df['R_3']=((df[c] - df.groupby('county')[c].transform('mean')) ** 2).sum(axis=1) ** 0.5

In [ ]:
df.head()

Although the `.apply()` method is more flexible and can handle complex operations, it is generally slower. On the other hand, the `.transform()` method can be much faster. When we design the procedures to use vector operations, we will realize significant performance benefits. 

**Note**: `Groupby.apply()` doesn't scale well with the number of groups, therefore this performance difference will be more pronounced with higher number of groups. 

## Pivot Table ##
Pivot tables allow us to summarize and aggregate large datasets into a more manageable format for analysis. When using `DataFrame.pivot_table()`, we provide the `index`, `columns`, and `values` arguments, as well as `aggfunc`. This will group the data based on `index` and `columns`, and perform the aggregation on `values`. We can apply multiple aggregation functions, which is generally faster and more memory-efficient than manual grouping and aggregation for large datasets. 

Below we create a pivot table that counts the number of each sex in each county. Furthermore, we derive the percentage of the total for each county. 

In [23]:
import polars as pl
df3 = pl.DataFrame({
    "county": ["A", "A", "B", "B", "B","B","C"],
    "sex": ["M", "F", "M", "F", "M","unk","unk"],
    "age": [25, 35, 40, 50, 60,24,24]
})

pivot_table = df3.pivot(
    values="age",
    index="county",
    # columns="sex",
    on="sex",
    aggregate_function="len"
)

print(pivot_table)


shape: (3, 4)
┌────────┬─────┬─────┬─────┐
│ county ┆ M   ┆ F   ┆ unk │
│ ---    ┆ --- ┆ --- ┆ --- │
│ str    ┆ u32 ┆ u32 ┆ u32 │
╞════════╪═════╪═════╪═════╡
│ A      ┆ 1   ┆ 1   ┆ 0   │
│ B      ┆ 2   ┆ 1   ┆ 1   │
│ C      ┆ 0   ┆ 0   ┆ 1   │
└────────┴─────┴─────┴─────┘


In [ ]:
%%cudf.pandas.line_profile
# DO NOT CHANGE THIS CELL

pvt_tbl=df[['county', 'sex', 'name']].pivot_table(index=['county'], columns=['sex'], values='name', aggfunc='count')
pvt_tbl=pvt_tbl.apply(lambda x: x/sum(x), axis=1)
display(pvt_tbl)

In [ ]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

**Well Done!** Let's move to the [next notebook](1-06_data_visualization.ipynb). 

<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">